# CHAR-RNN Training on Trump Speeches

This notebook trains a Character-RNN model on the cleaned Trump speeches dataset.

In [2]:
!pip install tensorflow

  Using cached tensorflow-2.20.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (4.5 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached werkzeug-3.1.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
Using cached tensorflow-2.20.0-cp313-cp313-macosx_12_0_arm64.whl (200.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 11.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/6

In [ ]:
import sys
import os
import pandas as pd
import tensorflow as tf
import tf_keras
from packaging import version

# Check versions
assert sys.version_info >= (3, 7)
assert version.parse(tf.__version__) >= version.parse("2.8.0")

In [3]:
import sys
import os
import pandas as pd

In [ ]:
import tensorflow as tf
import tf_keras

## 1. Load the Dataset

In [4]:
# Load the parquet file
df = pd.read_parquet('../../data/transcriptions.parquet')

# Use the specified column
text_column = 'clean-v1-with-stopwords'

# Concatenate all text
full_text = " ".join(df[text_column].dropna().astype(str).tolist())

print(f"Total text length: {len(full_text)} characters")
print(f"Sample: {full_text[:80]}")

Total text length: 37013060 characters
Sample: well that be good timing be not it we have to get that right we have to get that


## 2. Preprocessing & Vectorization

In [5]:
# Create a layer to map characters to integers
text_vec_layer = tf.keras.layers.TextVectorization(split="character", standardize="lower")
text_vec_layer.adapt([full_text])

# Encode the text
encoded = text_vec_layer([full_text])[0]

# Drop tokens 0 (pad) and 1 (unknown) as we won't use them
encoded -= 2 
n_tokens = text_vec_layer.vocabulary_size() - 2 
dataset_size = len(encoded)

print(f"Number of distinct tokens: {n_tokens}")

NameError: name 'tf' is not defined

## 3. Dataset Creation Helper

In [ ]:
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window_ds: window_ds.batch(length + 1))
    if shuffle:
        ds = ds.shuffle(100_000, seed=seed)
    ds = ds.batch(batch_size)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

# Create Training, Validation, and Test sets
length = 100
tf.random.set_seed(42)

print("Building datasets...")
# Using 90% for training, 5% for validation, 5% for test (approx)
train_size = int(dataset_size * 0.9)
valid_size = int(dataset_size * 0.05)

train_set = to_dataset(encoded[:train_size], length=length, shuffle=True, seed=42)
valid_set = to_dataset(encoded[train_size:train_size+valid_size], length=length)
test_set = to_dataset(encoded[train_size+valid_size:], length=length)

## 4. Build and Train the Model

In [ ]:
print("Building model...")
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "my_trump_model", monitor="val_accuracy", save_best_only=True
)

print("Starting training (this may take a while)...")
# Note: Training can take 1-2 hours on GPU. Reduce epochs for testing.
history = model.fit(train_set, validation_data=valid_set, epochs=10, callbacks=[model_ckpt])

# Wrap the model for easier inference later
final_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda X: X - 2),  # no <PAD> or <UNK> tokens
    model
])

## 5. Inference & Text Generation

In [ ]:
def next_char(text, temperature=1):
    """
    Predicts the next character. 
    Temperature controls randomness: 
    Low (near 0) = predictable/repetitive. High (>1) = random/chaotic.
    """
    y_proba = final_model.predict([text])[0, -1:]
    rescaled_logits = tf.math.log(y_proba) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
    return text_vec_layer.get_vocabulary()[char_id + 2]

def extend_text(text, n_chars=50, temperature=1):
    """Generates a sequence of characters."""
    print(f"Generating {n_chars} chars with temp {temperature}...")
    for _ in range(n_chars):
        text += next_char(text, temperature)
    return text

# --- Run Generation ---
tf.random.set_seed(42)

# 1. Test single prediction
print("Prediction test (We will make America...):")
y_proba = final_model.predict(["We will make America"])[0, -1]
y_pred = tf.argmax(y_proba)
print(f"Predicted next char: {text_vec_layer.get_vocabulary()[y_pred + 2]}")

# 2. Generate text with different temperatures
print("\n--- Temperature 0.01 (Conservative) ---")
print(extend_text("We will make America", temperature=0.01))

print("\n--- Temperature 1 (Balanced) ---")
print(extend_text("We will make America", temperature=1))

print("\n--- Temperature 100 (Chaotic) ---")
print(extend_text("We will make America", temperature=100))